## 1. Setup, Drive Mount & Create Images Folder

We mount Drive to reach the datasets, install the extra ML libraries not pre-installed on Colab (`lightgbm`, `xgboost`, `imbalanced-learn`), and create the `/content/images` folder that every plot in this notebook will be saved into.

In [ ]:
# Mount Google Drive so the CASHNET dataset folder becomes accessible
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install libraries not bundled by default in Colab
!pip install -q lightgbm xgboost imbalanced-learn

In [ ]:
# Core imports used throughout the notebook
import os
import gc
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

# Professional plotting defaults used for every figure in the notebook
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries loaded successfully.")

In [ ]:
# Create the folder that will hold every plot produced by this notebook
IMAGES_DIR = "/content/images"
os.makedirs(IMAGES_DIR, exist_ok=True)


def save_fig(fig, filename, tight=True):
    """Save a matplotlib figure as a high-quality PNG inside IMAGES_DIR.

    A single helper is used everywhere so every plot in the notebook is saved
    the same way (same DPI, same bbox handling) and the naming convention
    stays consistent across sections.
    """
    path = os.path.join(IMAGES_DIR, filename)
    if tight:
        fig.savefig(path, dpi=300, bbox_inches="tight")
    else:
        fig.savefig(path, dpi=300)
    print(f"Saved figure -> {path}")


print(f"Images will be saved to: {IMAGES_DIR}")

## 2. Load All Datasets (with smart sampling if files are large)

Each dataset is loaded through a single `load_csv_efficient()` helper so sampling fraction, row limits and dtype-downcasting are controlled consistently. PaySim and IEEE are large enough that we sample a fraction of rows to keep everything comfortably inside Colab's RAM budget while still preserving the class balance of the full data (sampling is done row-wise, not by filtering fraud/non-fraud, so the fraud rate is preserved).

In [ ]:
# Root folder holding every dataset, and the specific file paths inside it
BASE_DIR = "/content/drive/MyDrive/CASHNET"
MODEL_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

PATHS = {
    "paysim": f"{BASE_DIR}/paysim1/paysim1.csv",
    "ieee_train_tx": f"{BASE_DIR}/ieee-fraud-detection/train_transaction.csv",
    "ieee_train_id": f"{BASE_DIR}/ieee-fraud-detection/train_identity.csv",
    "ibm_hi_small": f"{BASE_DIR}/ibm-transactions/HI-Small_Trans.csv",
    "bank_outlets": f"{BASE_DIR}/Bank Outlets & ATM/Banking Export Data CSV.csv",
}

In [ ]:
def load_csv_efficient(path, sample_frac=None, nrows=None, dtype_downcast=True, label=""):
    """Load a CSV with optional row sampling and numeric dtype downcasting.

    Sampling is applied after the full read (pandas cannot sample while
    streaming a CSV), but downcasting floats/ints to smaller dtypes right
    afterwards keeps memory usage low for the rest of the notebook.
    """
    if not os.path.exists(path):
        print(f"[MISSING] {label or path}")
        return pd.DataFrame()

    print(f"Loading {label or path} ...")
    df = pd.read_csv(path, nrows=nrows, low_memory=False)

    if sample_frac is not None and 0 < sample_frac < 1:
        df = df.sample(frac=sample_frac, random_state=RANDOM_STATE).reset_index(drop=True)

    if dtype_downcast:
        for c in df.select_dtypes(include=["float64"]).columns:
            df[c] = pd.to_numeric(df[c], downcast="float")
        for c in df.select_dtypes(include=["int64"]).columns:
            df[c] = pd.to_numeric(df[c], downcast="integer")

    print(f"  -> shape: {df.shape}")
    return df

In [ ]:
# PaySim1: primary modeling dataset. Sampled to 40% of rows to control memory.
paysim_df = load_csv_efficient(PATHS["paysim"], sample_frac=0.4, label="PaySim1")

In [ ]:
# IEEE-CIS: profiled for EDA only. train_transaction and train_identity are
# merged on TransactionID (identity rows are optional/left-joined).
ieee_tx = load_csv_efficient(PATHS["ieee_train_tx"], sample_frac=0.5, label="IEEE train_transaction")
ieee_id = load_csv_efficient(PATHS["ieee_train_id"], label="IEEE train_identity")

if not ieee_tx.empty:
    ieee_df = ieee_tx.merge(ieee_id, on="TransactionID", how="left") if not ieee_id.empty else ieee_tx
else:
    ieee_df = pd.DataFrame()

del ieee_tx, ieee_id
gc.collect()
print("ieee_df shape:", ieee_df.shape)

In [ ]:
# IBM AML transactions: profiled for EDA only (different schema/label semantics).
ibm_df = load_csv_efficient(PATHS["ibm_hi_small"], label="IBM HI-Small_Trans")

In [ ]:
# Bank Outlets & ATM data: geospatial reference table used in Section 8.
bank_outlets_df = load_csv_efficient(PATHS["bank_outlets"], label="Bank Outlets & ATM")
bank_outlets_df.columns = [c.strip().lower().replace(" ", "_") for c in bank_outlets_df.columns]
bank_outlets_df.head()

## 3. Comprehensive Exploratory Data Analysis

This section covers, for the four datasets: structural summaries, missing-value analysis (with a visualization), class-imbalance analysis, distribution/box plots for outlier inspection, a correlation heatmap, time-based fraud patterns and a geospatial view of the outlet/ATM network. Every plot is saved to `/content/images/`.

In [ ]:
def eda_summary(df, name):
    """Print a compact structural + missing-value summary for one dataframe."""
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    print("Shape:", df.shape)
    print("\nDtype counts:\n", df.dtypes.value_counts())
    miss = df.isna().mean().sort_values(ascending=False)
    print("\nTop missing-value columns (%):\n", (miss[miss > 0] * 100).round(2).head(10))
    return miss


paysim_missing = eda_summary(paysim_df, "PaySim1")
ieee_missing = eda_summary(ieee_df, "IEEE-CIS Fraud")
ibm_missing = eda_summary(ibm_df, "IBM AML Transactions")
outlets_missing = eda_summary(bank_outlets_df, "Bank Outlets & ATM")

### 3.1 Missing value visualization

A heatmap of missingness (sampled for readability on wide tables) makes it easy to spot columns or row-ranges with systematic gaps, which is more actionable than a table of percentages alone.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.heatmap(paysim_df.isna(), cbar=False, yticklabels=False, ax=axes[0], cmap="rocket_r")
axes[0].set_title("PaySim1 — missing value map")

ieee_sample = ieee_df.sample(min(5000, len(ieee_df)), random_state=RANDOM_STATE) if not ieee_df.empty else ieee_df
sns.heatmap(ieee_sample.isna(), cbar=False, yticklabels=False, ax=axes[1], cmap="rocket_r")
axes[1].set_title("IEEE-CIS (5k row sample) — missing value map")

plt.tight_layout()
save_fig(fig, "03_missing_values_heatmap.png")
plt.show()

### 3.2 Class imbalance analysis

Fraud detection datasets are almost always heavily imbalanced. Quantifying the exact imbalance ratio up front determines whether we need SMOTE / class weighting later, and sets expectations for which metrics (PR-AUC over plain accuracy) matter most.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.countplot(x=paysim_df["isFraud"], ax=axes[0])
axes[0].set_title("PaySim1 — fraud class balance")
axes[0].set_xlabel("isFraud")
for p in axes[0].patches:
    axes[0].annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2, p.get_height()),
                      ha="center", va="bottom")

if not ieee_df.empty and "isFraud" in ieee_df.columns:
    sns.countplot(x=ieee_df["isFraud"], ax=axes[1])
    axes[1].set_title("IEEE-CIS — fraud class balance")
    for p in axes[1].patches:
        axes[1].annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2, p.get_height()),
                          ha="center", va="bottom")

plt.tight_layout()
save_fig(fig, "01_class_imbalance.png")
plt.show()

paysim_fraud_rate = paysim_df["isFraud"].mean()
print(f"PaySim1 fraud rate: {paysim_fraud_rate:.4%}  "
      f"(imbalance ratio ~1:{int((1 - paysim_fraud_rate) / max(paysim_fraud_rate, 1e-9)):,})")

### 3.3 Transaction type & amount distributions

Fraud in PaySim concentrates almost exclusively in `CASH_OUT` and `TRANSFER` transaction types, which is the empirical justification for the withdrawal-focused feature engineering in Section 5.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=paysim_df, x="type", hue="isFraud", ax=axes[0])
axes[0].set_title("Transaction type vs fraud")
axes[0].tick_params(axis="x", rotation=25)

sns.histplot(np.log1p(paysim_df["amount"]), bins=60, kde=True, ax=axes[1], color="steelblue")
axes[1].set_title("log(1 + amount) distribution")
axes[1].set_xlabel("log(1 + amount)")

plt.tight_layout()
save_fig(fig, "02_paysim_type_amount_distribution.png")
plt.show()

### 3.4 Outlier detection (IQR method)

Boxplots of transaction amount, split by fraud label, both flag outliers visually and justify the IQR-based capping applied in Section 4 (fraud transactions naturally sit further into the tail, so outliers are winsorized rather than dropped to avoid discarding real fraud signal).

In [ ]:
def iqr_bounds(series, k=1.5):
    """Return the lower/upper Tukey fences for a numeric series."""
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr


lower, upper = iqr_bounds(paysim_df["amount"])
n_outliers = ((paysim_df["amount"] < lower) | (paysim_df["amount"] > upper)).sum()
print(f"IQR bounds for amount: [{lower:,.2f}, {upper:,.2f}]  |  outliers flagged: {n_outliers:,}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.boxplot(data=paysim_df, x="isFraud", y="amount", ax=axes[0])
axes[0].set_yscale("log")
axes[0].set_title("Amount by fraud label (log scale)")

sns.boxplot(data=paysim_df, y="amount", ax=axes[1])
axes[1].set_yscale("log")
axes[1].set_title("Overall amount distribution (log scale)")

plt.tight_layout()
save_fig(fig, "05_outlier_boxplots.png")
plt.show()

### 3.5 Correlation heatmap

A correlation heatmap over the numeric PaySim columns highlights the near-identity relationships between balance fields that later drive the `errorBalanceOrig` / `errorBalanceDest` engineered features (large deviations from the expected balance identity are a classic fraud tell).

In [ ]:
numeric_cols = paysim_df.select_dtypes(include=[np.number]).columns
corr = paysim_df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("PaySim1 — numeric feature correlation heatmap")
plt.tight_layout()
save_fig(fig, "04_correlation_heatmap.png")
plt.show()

### 3.6 Time-based analysis

PaySim's `step` field represents an hour index over a 30-day simulation. Converting it to hour-of-day lets us check whether fraud clusters at particular times, which feeds the `hour_of_day` / `is_night` features in Section 5.

In [ ]:
paysim_df["hour_of_day_eda"] = paysim_df["step"] % 24
hourly_fraud_rate = paysim_df.groupby("hour_of_day_eda")["isFraud"].mean()

fig, ax = plt.subplots(figsize=(10, 5))
hourly_fraud_rate.plot(kind="bar", ax=ax, color="firebrick")
ax.set_title("Fraud rate by hour of day")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Fraud rate")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.tight_layout()
save_fig(fig, "07_time_based_fraud_pattern.png")
plt.show()

paysim_df.drop(columns=["hour_of_day_eda"], inplace=True)

### 3.7 Geospatial analysis of the outlet / ATM network

The Bank Outlets & ATM table is plotted as-is to understand branch density and coverage before it is turned into risk zones in Section 8.

In [ ]:
lat_col = next((c for c in bank_outlets_df.columns if "lat" in c), None)
lon_col = next((c for c in bank_outlets_df.columns if "lon" in c or "lng" in c), None)

if lat_col and lon_col:
    geo_df = bank_outlets_df.dropna(subset=[lat_col, lon_col])

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].scatter(geo_df[lon_col], geo_df[lat_col], s=8, alpha=0.5, color="teal")
    axes[0].set_title("Bank outlet / ATM locations")
    axes[0].set_xlabel("longitude")
    axes[0].set_ylabel("latitude")

    sns.kdeplot(x=geo_df[lon_col], y=geo_df[lat_col], fill=True, cmap="rocket", ax=axes[1], thresh=0.05)
    axes[1].set_title("Outlet density heatmap")
    axes[1].set_xlabel("longitude")
    axes[1].set_ylabel("latitude")

    plt.tight_layout()
    save_fig(fig, "08_geospatial_outlets.png")
    plt.show()
else:
    print("No latitude/longitude columns detected in bank outlets data — geospatial plots skipped.")

## 4. Advanced Data Cleaning & Preprocessing

Cleaning is implemented as small reusable functions (duplicate removal, missing-value imputation, outlier capping) so the exact same logic can later be reapplied to a single new transaction inside the inference pipeline in Section 9.

In [ ]:
def drop_high_missing(df, threshold=0.9):
    """Drop columns whose missing-value fraction exceeds `threshold`."""
    keep_cols = df.columns[df.isna().mean() < threshold]
    dropped = set(df.columns) - set(keep_cols)
    if dropped:
        print(f"Dropping {len(dropped)} columns with >{threshold:.0%} missing values")
    return df[keep_cols]


def fill_missing(df):
    """Impute numeric columns with -999 (out-of-range sentinel) and
    categorical columns with the literal string 'missing', so missingness
    itself remains a usable signal for tree-based models."""
    df = df.copy()
    num_cols = df.select_dtypes(include=[np.number]).columns
    cat_cols = df.select_dtypes(exclude=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(-999)
    df[cat_cols] = df[cat_cols].fillna("missing")
    return df


def cap_outliers(df, column, k=1.5):
    """Winsorize a numeric column to the Tukey IQR fences instead of
    dropping rows, which preserves fraud cases that legitimately sit in
    the tail of the amount distribution."""
    lower, upper = iqr_bounds(df[column], k=k)
    df = df.copy()
    df[column] = df[column].clip(lower=max(lower, 0), upper=upper)
    return df

In [ ]:
print("Cleaning PaySim1 ...")
before = paysim_df.shape[0]
paysim_df = paysim_df.drop_duplicates()
print(f"  Removed {before - paysim_df.shape[0]:,} duplicate rows")

paysim_df = fill_missing(paysim_df)
paysim_df["isFraud"] = paysim_df["isFraud"].astype(int)
paysim_df = cap_outliers(paysim_df, "amount")
print("PaySim1 cleaned shape:", paysim_df.shape)

In [ ]:
if not ieee_df.empty:
    print("Cleaning IEEE-CIS ...")
    ieee_df = drop_high_missing(ieee_df, threshold=0.9)
    ieee_df = fill_missing(ieee_df)
    print("IEEE-CIS cleaned shape:", ieee_df.shape)

In [ ]:
if not ibm_df.empty:
    print("Cleaning IBM AML ...")
    ibm_df = drop_high_missing(ibm_df, threshold=0.9)
    ibm_df = fill_missing(ibm_df)
    print("IBM AML cleaned shape:", ibm_df.shape)

In [ ]:
print("Cleaning Bank Outlets & ATM ...")
if lat_col and lon_col:
    before = bank_outlets_df.shape[0]
    bank_outlets_df = bank_outlets_df.dropna(subset=[lat_col, lon_col]).reset_index(drop=True)
    print(f"  Dropped {before - bank_outlets_df.shape[0]:,} rows missing coordinates")
bank_outlets_df = fill_missing(bank_outlets_df)
print("Bank Outlets cleaned shape:", bank_outlets_df.shape)

## 5. Feature Engineering

Engineered features are grouped by purpose: time-based, cash-out/withdrawal-specific, fraud-pattern (balance-identity errors, merchant flags), account-velocity aggregates, and geospatial zoning for the outlet data. The same `engineer_paysim_features()` function is reused verbatim inside the final inference pipeline in Section 9, guaranteeing train/inference consistency.

In [ ]:
def engineer_paysim_features(df):
    """Create all model features from a raw (or single-row) PaySim-style
    transaction frame. Kept as one pure function so training and inference
    apply exactly the same transformation."""
    df = df.copy()

    # --- time-based features ---
    df["hour_of_day"] = df["step"] % 24
    df["day"] = df["step"] // 24
    df["is_night"] = (df["hour_of_day"] < 6).astype(int)

    # --- cash-out / withdrawal features ---
    df["is_cash_out"] = (df["type"] == "CASH_OUT").astype(int)
    df["is_transfer"] = (df["type"] == "TRANSFER").astype(int)
    df["is_withdrawal_like"] = (df["is_cash_out"] | (df["type"] == "CASH_IN").astype(int)).astype(int)

    # --- fraud-pattern / balance-identity features ---
    # In a legitimate transaction, oldbalance - amount should equal newbalance.
    # Large deviations from that identity are a strong fraud signal.
    df["errorBalanceOrig"] = df["newbalanceOrig"] + df["amount"] - df["oldbalanceOrg"]
    df["errorBalanceDest"] = df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]
    df["orig_balance_zero"] = (df["oldbalanceOrg"] == 0).astype(int)
    df["dest_balance_zero"] = (df["oldbalanceDest"] == 0).astype(int)
    df["amount_to_balance_ratio"] = df["amount"] / (df["oldbalanceOrg"] + 1)
    df["dest_is_merchant"] = df["nameDest"].astype(str).str.startswith("M").astype(int)

    # --- account velocity / aggregate features ---
    orig_counts = df.groupby("nameOrig")["amount"].transform("count")
    orig_avg_amount = df.groupby("nameOrig")["amount"].transform("mean")
    df["orig_txn_count"] = orig_counts
    df["orig_avg_amount"] = orig_avg_amount
    df["amount_deviation_from_avg"] = df["amount"] - df["orig_avg_amount"]

    return df


print("Engineering PaySim features ...")
paysim_fe = engineer_paysim_features(paysim_df)
print("paysim_fe shape:", paysim_fe.shape)

In [ ]:
# Numeric features go through scaling, the categorical transaction "type"
# goes through one-hot encoding — both handled later by a ColumnTransformer.
NUMERIC_FEATURES = [
    "amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest",
    "hour_of_day", "day", "is_night", "is_cash_out", "is_transfer", "is_withdrawal_like",
    "errorBalanceOrig", "errorBalanceDest", "orig_balance_zero", "dest_balance_zero",
    "amount_to_balance_ratio", "dest_is_merchant", "orig_txn_count", "orig_avg_amount",
    "amount_deviation_from_avg",
]
CATEGORICAL_FEATURES = ["type"]
FEATURE_COLS = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET_COL = "isFraud"

print(f"{len(FEATURE_COLS)} model features selected "
      f"({len(NUMERIC_FEATURES)} numeric, {len(CATEGORICAL_FEATURES)} categorical)")

In [ ]:
# Geospatial zoning: cluster outlets/ATMs into risk zones with KMeans so
# every branch belongs to a small number of manageable geographic groups.
if lat_col and lon_col:
    n_zones = min(15, max(2, bank_outlets_df.shape[0] // 20))
    zone_model = KMeans(n_clusters=n_zones, random_state=RANDOM_STATE, n_init=10)
    bank_outlets_df["zone_id"] = zone_model.fit_predict(bank_outlets_df[[lat_col, lon_col]])
    zone_density = bank_outlets_df["zone_id"].value_counts().rename("zone_outlet_count")
    bank_outlets_df = bank_outlets_df.join(zone_density, on="zone_id")
    print(f"Clustered outlets into {n_zones} geospatial zones")
else:
    zone_model = None
    print("Skipping zone clustering: no lat/lon columns available.")

## 6. Machine Learning Pipeline (Training + Evaluation)

A stratified train/validation/test split is used (train for fitting, validation for model comparison and hyperparameter tuning, test held out for the final, unbiased evaluation). Preprocessing (scaling + one-hot encoding) and class-imbalance handling (SMOTE) are wrapped inside a single `imblearn` `Pipeline` per model, so no leakage occurs between folds and the exact same object can be persisted and reused at inference time.

In [ ]:
X = paysim_fe[FEATURE_COLS]
y = paysim_fe[TARGET_COL]

# 60% train / 20% validation / 20% test, all stratified on the target so
# the (rare) fraud class keeps the same proportion in every split.
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape}, Validation: {X_val.shape}, Test: {X_test.shape}")
print(f"Train fraud rate: {y_train.mean():.4%} | Val: {y_val.mean():.4%} | Test: {y_test.mean():.4%}")

In [ ]:
# ColumnTransformer keeps scaling and encoding declarative and leak-free:
# it is fit only on the training fold inside each pipeline's .fit() call.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ]
)

# SMOTE is applied only inside the training fold of each pipeline (never on
# validation/test), and only brings the minority class up to 10% of the
# majority rather than full 50/50 balance, which is far cheaper to fit and
# avoids overwhelming the model with synthetic fraud examples.
smote = SMOTE(sampling_strategy=0.1, random_state=RANDOM_STATE)

pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print("scale_pos_weight for boosted trees:", round(pos_weight, 2))

In [ ]:
def build_pipeline(estimator):
    """Wrap preprocessing + SMOTE + a classifier into a single fit-predict object."""
    return ImbPipeline(steps=[
        ("preprocessor", preprocessor),
        ("smote", smote),
        ("classifier", estimator),
    ])


model_pipelines = {
    "LogisticRegression": build_pipeline(
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
    ),
    "RandomForest": build_pipeline(
        RandomForestClassifier(n_estimators=200, max_depth=12, class_weight="balanced",
                                n_jobs=-1, random_state=RANDOM_STATE)
    ),
    "XGBoost": build_pipeline(
        XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.08,
                       scale_pos_weight=pos_weight, eval_metric="aucpr",
                       n_jobs=-1, random_state=RANDOM_STATE)
    ),
    "LightGBM": build_pipeline(
        LGBMClassifier(n_estimators=300, max_depth=-1, learning_rate=0.08,
                        class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
    ),
}

print("Model pipelines ready:", list(model_pipelines.keys()))

In [ ]:
def evaluate_pipeline(name, pipeline, X_eval, y_eval):
    """Fit-independent evaluation: compute the full metric suite for one
    already-fitted pipeline on a given split."""
    proba = pipeline.predict_proba(X_eval)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {
        "accuracy": accuracy_score(y_eval, pred),
        "precision": precision_score(y_eval, pred, zero_division=0),
        "recall": recall_score(y_eval, pred, zero_division=0),
        "f1": f1_score(y_eval, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_eval, proba),
        "pr_auc": average_precision_score(y_eval, proba),
        "proba": proba,
        "pred": pred,
    }


val_results = {}
for name, pipe in model_pipelines.items():
    print(f"Training {name} ...")
    t0 = time.time()
    pipe.fit(X_train, y_train)
    val_results[name] = evaluate_pipeline(name, pipe, X_val, y_val)
    dt = time.time() - t0
    r = val_results[name]
    print(f"  {name:18s} ROC-AUC={r['roc_auc']:.4f}  PR-AUC={r['pr_auc']:.4f}  "
          f"F1={r['f1']:.4f}  ({dt:.1f}s)")

### 6.1 Hyperparameter tuning

The strongest validation model (by PR-AUC) is tuned further with `RandomizedSearchCV` over a small, sensible search space. `RandomizedSearchCV` is used instead of a full grid search to keep runtime bounded on Colab; swapping in `optuna` with the same parameter ranges is a drop-in alternative if a more exhaustive search is later desired.

In [ ]:
candidate_for_tuning = max(val_results, key=lambda k: val_results[k]["pr_auc"])
print("Candidate selected for tuning:", candidate_for_tuning)

param_distributions = {
    "RandomForest": {
        "classifier__n_estimators": [150, 250, 350],
        "classifier__max_depth": [8, 12, 16, None],
        "classifier__min_samples_leaf": [1, 2, 5],
    },
    "XGBoost": {
        "classifier__n_estimators": [200, 300, 400],
        "classifier__max_depth": [4, 6, 8],
        "classifier__learning_rate": [0.03, 0.08, 0.15],
        "classifier__subsample": [0.7, 0.85, 1.0],
    },
    "LightGBM": {
        "classifier__n_estimators": [200, 300, 400],
        "classifier__num_leaves": [31, 63, 127],
        "classifier__learning_rate": [0.03, 0.08, 0.15],
    },
    "LogisticRegression": {
        "classifier__C": [0.01, 0.1, 1.0, 10.0],
    },
}

search_space = param_distributions.get(candidate_for_tuning)
if search_space:
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    search = RandomizedSearchCV(
        model_pipelines[candidate_for_tuning],
        param_distributions=search_space,
        n_iter=8,
        scoring="average_precision",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0,
    )
    print(f"Running RandomizedSearchCV for {candidate_for_tuning} ...")
    search.fit(X_train, y_train)
    print("Best params:", search.best_params_)
    model_pipelines[candidate_for_tuning] = search.best_estimator_
    val_results[candidate_for_tuning] = evaluate_pipeline(
        candidate_for_tuning, model_pipelines[candidate_for_tuning], X_val, y_val
    )
    print(f"Tuned {candidate_for_tuning} PR-AUC on validation: "
          f"{val_results[candidate_for_tuning]['pr_auc']:.4f}")
else:
    print("No search space defined for this model; skipping tuning.")

## 7. Model Comparison & Best Model Selection

All four pipelines are compared on the held-out **test** set (never touched during training or tuning) using the full metric suite, ROC and Precision-Recall curves, a confusion matrix for the winner, and a feature-importance plot. PR-AUC is used as the primary selection criterion because with a fraud rate under 1%, ROC-AUC can look deceptively strong even for a weak model.

In [ ]:
test_results = {}
for name, pipe in model_pipelines.items():
    test_results[name] = evaluate_pipeline(name, pipe, X_test, y_test)

comparison_table = pd.DataFrame({
    name: {
        "Accuracy": r["accuracy"], "Precision": r["precision"], "Recall": r["recall"],
        "F1": r["f1"], "ROC-AUC": r["roc_auc"], "PR-AUC": r["pr_auc"],
    }
    for name, r in test_results.items()
}).T.sort_values("PR-AUC", ascending=False)

print("Test-set model comparison:")
comparison_table.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
comparison_table[["ROC-AUC", "PR-AUC", "F1"]].plot(kind="bar", ax=ax)
ax.set_title("Model comparison on held-out test set")
ax.set_ylabel("Score")
ax.legend(loc="lower right")
plt.tight_layout()
save_fig(fig, "09_model_comparison_bar.png")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, r in test_results.items():
    fpr, tpr, _ = roc_curve(y_test, r["proba"])
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={r['roc_auc']:.3f})")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="grey")
axes[0].set_title("ROC curve comparison")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()

for name, r in test_results.items():
    prec, rec, _ = precision_recall_curve(y_test, r["proba"])
    axes[1].plot(rec, prec, label=f"{name} (AP={r['pr_auc']:.3f})")
axes[1].set_title("Precision-Recall curve comparison")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()

plt.tight_layout()
save_fig(fig, "10_roc_pr_curve_comparison.png")
plt.show()

In [ ]:
best_name = comparison_table.index[0]
best_pipeline = model_pipelines[best_name]
print(f"Best model selected: {best_name}")
print(classification_report(y_test, test_results[best_name]["pred"]))

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
cm = confusion_matrix(y_test, test_results[best_name]["pred"])
ConfusionMatrixDisplay(cm, display_labels=["Legit", "Fraud"]).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion matrix — {best_name}")
plt.tight_layout()
save_fig(fig, "11_confusion_matrix_best_model.png")
plt.show()

In [ ]:
classifier_step = best_pipeline.named_steps["classifier"]
if hasattr(classifier_step, "feature_importances_"):
    ohe_cats = best_pipeline.named_steps["preprocessor"].named_transformers_["cat"].get_feature_names_out(CATEGORICAL_FEATURES)
    all_feature_names = NUMERIC_FEATURES + list(ohe_cats)
    importances = pd.Series(classifier_step.feature_importances_, index=all_feature_names)
    importances = importances.sort_values(ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.barplot(x=importances.values, y=importances.index, ax=ax, color="darkorange")
    ax.set_title(f"Top 15 feature importances — {best_name}")
    plt.tight_layout()
    save_fig(fig, "12_feature_importance.png")
    plt.show()
else:
    print(f"{best_name} does not expose feature_importances_ (e.g. Logistic Regression); skipping plot.")

## 8. Withdrawal Hotspot Prediction & Geospatial Analysis

A withdrawal risk score blends the model's fraud probability with cash-out-specific signals. Because PaySim has no real geo-coordinates, every scored transaction is mapped to one of the KMeans outlet zones from Section 5 via a deterministic hash of the account id — a placeholder join key documented clearly so it can be swapped for a real account-to-branch mapping without touching the scoring logic. Zone-level mean risk is then ranked to surface the top high-risk locations, visualized as a risk heatmap over the outlet network.

In [ ]:
def compute_withdrawal_risk(df, fraud_proba):
    """Blend the model's fraud probability with cash-out-specific signals
    into a single 0-1 withdrawal risk score."""
    risk = fraud_proba.copy()
    if "is_cash_out" in df.columns:
        risk = risk + 0.1 * df["is_cash_out"].values
    if "amount_deviation_from_avg" in df.columns:
        dev = df["amount_deviation_from_avg"].values
        dev_norm = (dev - dev.min()) / (dev.max() - dev.min() + 1e-9)
        risk = risk + 0.1 * dev_norm
    if "orig_balance_zero" in df.columns:
        risk = risk + 0.05 * df["orig_balance_zero"].values
    return np.clip(risk, 0, 1)


withdrawal_risk_scores = compute_withdrawal_risk(paysim_fe.loc[X_test.index], test_results[best_name]["proba"])
print("Sample withdrawal risk scores:", np.round(withdrawal_risk_scores[:10], 3))

In [ ]:
def assign_zone(name_orig_series, n_zones):
    """Deterministically map an account id to one of n_zones. Placeholder
    for a real account -> branch/ATM mapping when one becomes available."""
    return name_orig_series.astype(str).apply(lambda s: abs(hash(s)) % n_zones)


def get_high_risk_zones(scored_df, outlets_df, zone_col="zone_id", top_n=10):
    """Rank geospatial zones by mean withdrawal risk and attach outlet metadata."""
    if zone_col not in outlets_df.columns:
        print("No zone information available; run the geospatial clustering step first.")
        return pd.DataFrame()
    n_zones = outlets_df[zone_col].nunique()
    zones = assign_zone(scored_df["nameOrig"], n_zones)
    zone_risk = scored_df.assign(zone_id=zones).groupby("zone_id")["withdrawal_risk"].mean()
    zone_risk = zone_risk.sort_values(ascending=False).head(top_n)
    outlet_lookup = outlets_df.drop_duplicates("zone_id").set_index("zone_id")
    hotspots = zone_risk.to_frame("mean_withdrawal_risk").join(outlet_lookup, how="left")
    return hotspots.reset_index()


scored_test = paysim_fe.loc[X_test.index, ["nameOrig"]].copy()
scored_test["withdrawal_risk"] = withdrawal_risk_scores

if zone_model is not None:
    hotspots_df = get_high_risk_zones(scored_test, bank_outlets_df, top_n=10)
    display_cols = [c for c in hotspots_df.columns
                     if c in ["zone_id", "mean_withdrawal_risk", "zone_outlet_count", lat_col, lon_col]]
    print("Top high-risk zones:")
    display(hotspots_df[display_cols])
else:
    hotspots_df = pd.DataFrame()
    print("Zone clustering unavailable; skipping hotspot ranking.")

In [ ]:
if not hotspots_df.empty:
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.scatter(bank_outlets_df[lon_col], bank_outlets_df[lat_col], s=8, alpha=0.15,
               color="grey", label="all outlets")
    hot = hotspots_df.dropna(subset=[lat_col, lon_col])
    sizes = 250 * hot["mean_withdrawal_risk"] / (hot["mean_withdrawal_risk"].max() + 1e-9)
    scatter = ax.scatter(hot[lon_col], hot[lat_col], s=sizes + 30, c=hot["mean_withdrawal_risk"],
                          cmap="Reds", edgecolor="black", linewidth=0.5, label="high-risk zones")
    plt.colorbar(scatter, ax=ax, label="mean withdrawal risk")
    ax.legend(loc="upper right")
    ax.set_title("Predicted withdrawal risk hotspots")
    ax.set_xlabel("longitude")
    ax.set_ylabel("latitude")
    plt.tight_layout()
    save_fig(fig, "13_withdrawal_risk_heatmap.png")
    plt.show()

    fig, ax = plt.subplots(figsize=(9, 5))
    top_plot = hotspots_df.sort_values("mean_withdrawal_risk", ascending=True)
    ax.barh(top_plot["zone_id"].astype(str), top_plot["mean_withdrawal_risk"], color="crimson")
    ax.set_title("Top 10 high-risk withdrawal zones")
    ax.set_xlabel("Mean withdrawal risk")
    ax.set_ylabel("Zone ID")
    plt.tight_layout()
    save_fig(fig, "14_top_high_risk_zones.png")
    plt.show()
else:
    print("Skipping hotspot visualization (no zone data).")

## 9. Final Inference Function

The trained pipeline, the withdrawal-risk logic and the zone lookup are wrapped into a single `CashFraudPipeline` class exposing one `predict()` method, so a new transaction can be scored end-to-end (fraud probability, withdrawal risk score, and suggested high-risk zones) with one call.

In [ ]:
class CashFraudPipeline:
    """Single entry point for scoring a new transaction end-to-end."""

    def __init__(self, fitted_pipeline, feature_cols, outlets_df=None, zone_col="zone_id"):
        self.fitted_pipeline = fitted_pipeline
        self.feature_cols = feature_cols
        self.outlets_df = outlets_df
        self.zone_col = zone_col

    def _prepare(self, transaction: dict) -> pd.DataFrame:
        row = pd.DataFrame([transaction])
        row = engineer_paysim_features(row)
        for col in self.feature_cols:
            if col not in row.columns:
                row[col] = 0
        return row[self.feature_cols]

    def predict(self, transaction: dict) -> dict:
        row = self._prepare(transaction)
        fraud_proba = float(self.fitted_pipeline.predict_proba(row)[:, 1][0])
        withdrawal_risk = float(compute_withdrawal_risk(row, np.array([fraud_proba]))[0])

        zones = []
        if self.outlets_df is not None and self.zone_col in self.outlets_df.columns:
            n_zones = self.outlets_df[self.zone_col].nunique()
            zone_id = int(abs(hash(str(transaction.get("nameOrig", "unknown")))) % n_zones)
            match = self.outlets_df[self.outlets_df[self.zone_col] == zone_id]
            zones = match.head(3).to_dict("records")

        return {
            "fraud_probability": round(fraud_proba, 4),
            "withdrawal_risk_score": round(withdrawal_risk, 4),
            "suggested_high_risk_zones": zones,
        }


pipeline = CashFraudPipeline(
    fitted_pipeline=best_pipeline,
    feature_cols=FEATURE_COLS,
    outlets_df=bank_outlets_df if zone_model is not None else None,
)
print("CashFraudPipeline ready.")

In [ ]:
# Example: score a brand new, previously unseen transaction
example_transaction = {
    "step": 10,
    "type": "CASH_OUT",
    "amount": 181000.0,
    "nameOrig": "C1231006815",
    "oldbalanceOrg": 181000.0,
    "newbalanceOrig": 0.0,
    "nameDest": "C1666544295",
    "oldbalanceDest": 0.0,
    "newbalanceDest": 0.0,
}

result = pipeline.predict(example_transaction)
print(json.dumps(result, indent=2, default=str))

## 10. Save Model, Pipeline & Summary Report

The fitted pipeline (preprocessing + SMOTE + classifier bundled together), the zone-clustering model, the enriched outlets table and a JSON summary of the final metrics are all persisted to Drive, so the notebook can be reloaded and reused in a fresh runtime without retraining.

In [ ]:
joblib.dump(best_pipeline, f"{MODEL_DIR}/best_cash_fraud_pipeline.joblib")
joblib.dump(FEATURE_COLS, f"{MODEL_DIR}/feature_cols.joblib")

if zone_model is not None:
    joblib.dump(zone_model, f"{MODEL_DIR}/zone_kmeans.joblib")
    bank_outlets_df.to_csv(f"{MODEL_DIR}/bank_outlets_with_zones.csv", index=False)

summary_report = {
    "best_model": best_name,
    "test_metrics": {k: round(v, 4) for k, v in comparison_table.loc[best_name].to_dict().items()},
    "all_models_compared": comparison_table.round(4).to_dict(orient="index"),
    "n_features": len(FEATURE_COLS),
    "train_rows": int(X_train.shape[0]),
    "val_rows": int(X_val.shape[0]),
    "test_rows": int(X_test.shape[0]),
    "fraud_rate_train": round(float(y_train.mean()), 6),
    "images_saved_to": IMAGES_DIR,
    "artifacts_saved_to": MODEL_DIR,
}

with open(f"{MODEL_DIR}/run_summary.json", "w") as f:
    json.dump(summary_report, f, indent=2)

print(json.dumps(summary_report, indent=2))
print(f"\nAll model artifacts saved to: {MODEL_DIR}")
print(f"All figures saved to: {IMAGES_DIR}")
print(f"Total images saved: {len(os.listdir(IMAGES_DIR))}")

### How to reload and reuse this pipeline later

```python
best_pipeline = joblib.load(f"{MODEL_DIR}/best_cash_fraud_pipeline.joblib")
FEATURE_COLS = joblib.load(f"{MODEL_DIR}/feature_cols.joblib")
bank_outlets_df = pd.read_csv(f"{MODEL_DIR}/bank_outlets_with_zones.csv")

pipeline = CashFraudPipeline(best_pipeline, FEATURE_COLS, outlets_df=bank_outlets_df)
result = pipeline.predict(new_transaction_dict)
print(result)
```
